# AIDA 1141 Project Exercise 08 – Error and Slice Review

**Question:** Which validation assessments does the model get wrong or place near its threshold, what do those cases have in common, and does performance differ across operational slices?

This notebook uses the saved Exercise 06 validation predictions and threshold; nothing is refitted and test data is not used. The assessment-time inputs are added from Exercise 02 to describe each case.

Simulation-only fields (base processing days, workload delay, hidden delay, and delay reason) are added for error analysis only. They were never model inputs and are not used to change the model. They support post-prediction review of patterns associated with errors, but do not establish why individual predictions failed. All results describe methods applied to simulated outcomes, not real employees or HR performance.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

EX02 = Path("../02-synthetic-request-records")
EX06 = Path("../06-model-training-and-split")
EX07 = Path("../07-evaluation-measures")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

read = lambda name: pd.read_csv(EX02 / "data" / name, keep_default_na=False, na_values=[""])
predictions = pd.read_csv(EX06 / "outputs" / "validation_predictions.csv")
settings = pd.read_csv(EX06 / "outputs" / "model_settings.csv", index_col=0)["value"]
MODEL_THRESHOLD = float(settings["threshold"])

INPUTS = ["business_days_until_start", "current_status", "business_days_in_current_status", "team_open_workload"]
REQUEST_FIELDS = ["task_type", "hiring_department", "employee_type", "intake_channel", "info_issue_at_intake"]
SIMULATION_FIELDS = ["sim_base_processing_days", "sim_workload_delay_days", "sim_hidden_delay_days", "sim_delay_reason"]
validation = (predictions
              .merge(read("assessments.csv")[["assessment_id"] + INPUTS], on="assessment_id", how="left",
                     validate="one_to_one")
              .merge(read("requests.csv")[["request_id"] + REQUEST_FIELDS], on="request_id", how="left",
                     validate="many_to_one")
              .merge(read("simulation_truth.csv")[["request_id"] + SIMULATION_FIELDS], on="request_id", how="left",
                     validate="many_to_one"))
ex07_confusion = pd.read_csv(EX07 / "outputs" / "confusion_matrices_assessments.csv", index_col=0)
print(f"Loaded {len(validation)} validation assessments; model threshold {MODEL_THRESHOLD:.4f} (rounded for display)")

Loaded 526 validation assessments; model threshold 0.1254 (rounded for display)


## A. Consistency checks

These checks confirm that every validation assessment matched its inputs and simulation values, that the saved threshold reproduces the model's flags, and that the confusion counts match Exercise 07. Each early-warning assessment is then labelled TP, FP, FN, or TN for the model and for the global days rule.

In [2]:
MODEL, RULE = "logistic_regression", "days_until_start_rule"

# Validate the True/False columns and probabilities before using them
boolean_ok = all(validation[c].dtype == bool for c in ["early_warning", "at_risk", MODEL, RULE])
probability_ok = validation["risk_probability"].notna().all() and validation["risk_probability"].between(0, 1).all()
assert boolean_ok and probability_ok, "Labels, flags, or probabilities are missing or invalid."

early = validation[validation["early_warning"]].copy()

def outcome(actual, flagged):
    """TP, FP, FN, or TN for each assessment."""
    return np.select([flagged & actual, flagged & ~actual, ~flagged & actual], ["TP", "FP", "FN"], default="TN")

early["model_outcome"] = outcome(early["at_risk"], early[MODEL])
early["rule_outcome"] = outcome(early["at_risk"], early[RULE])
model_counts = early["model_outcome"].value_counts().reindex(["TP", "FP", "FN", "TN"])
rule_counts = early["rule_outcome"].value_counts().reindex(["TP", "FP", "FN", "TN"])

checks = pd.Series({
    "526 validation assessments, one row each": len(validation) == 526 and validation["assessment_id"].is_unique,
    "Every assessment matched its inputs and simulation values":
        validation[INPUTS + REQUEST_FIELDS + SIMULATION_FIELDS].notna().all().all(),
    "Label, early-warning, and flag columns are True/False only": boolean_ok,
    "Every probability is present and between 0 and 1": probability_ok,
    "Saved threshold reproduces the model's flags":
        ((validation["risk_probability"] >= MODEL_THRESHOLD) == validation[MODEL]).all(),
    "Confusion counts match Exercise 07":
        model_counts.tolist() == ex07_confusion.loc[MODEL, ["TP", "FP", "FN", "TN"]].tolist()
        and rule_counts.tolist() == ex07_confusion.loc[RULE, ["TP", "FP", "FN", "TN"]].tolist(),
}, name="passed")
assert checks.all(), "Inputs do not match Exercises 06 and 07."
checks.to_frame()

,passed
"526 validation assessments, one row each",True
Every assessment matched its inputs and simulation values,True
"Label, early-warning, and flag columns are True/False only",True
Every probability is present and between 0 and 1,True
Saved threshold reproduces the model's flags,True
Confusion counts match Exercise 07,True


## B. Error-review table

Representative validation early-warning assessments are selected from six groups:

1. **Missed at-risk:** every model false negative.
2. **Highest-scoring false positives:** the five not-at-risk assessments with the highest estimated risk.
3. **Boundary band, just above** and 4. **just below the threshold:** cases within a predefined score band around the decision threshold are selected for boundary review. The band runs from half to double the threshold. It is exploratory and does not represent a confidence interval or the final review-queue policy.
5. **Rule flagged, model did not, not at risk:** model–rule disagreements where the model avoided a false positive the rule made, choosing the lowest estimated risk.
6. **Correct contrast:** the two highest-scoring true positives.

Cases are deliberately selected as representative examples of each review group, to illustrate particular patterns. They are not a random or statistically representative sample of all assessments, and some assessments appear in more than one group. Ties in the sorting score are broken by assessment ID, so the selection is reproducible.

Each row shows the assessment-time inputs, the model's estimated risk and flag, the global rule's flag, the simulated outcome, and the simulation-only audit fields used for analysis.

In [3]:
band_low, band_high = MODEL_THRESHOLD / 2, MODEL_THRESHOLD * 2
early["in_boundary_band"] = early["risk_probability"].between(band_low, band_high, inclusive="left")
early["model_rule_disagree"] = early[MODEL] != early[RULE]

group_sizes = pd.Series({
    "Missed at-risk (model false negatives)": (early["model_outcome"] == "FN").sum(),
    "Model false positives": (early["model_outcome"] == "FP").sum(),
    f"Boundary-review band ({band_low:.3f} to {band_high:.3f})": early["in_boundary_band"].sum(),
    "Model and global rule disagree": early["model_rule_disagree"].sum(),
}, name="assessments")
disagreements = (early[early["model_rule_disagree"]].groupby(["at_risk", MODEL]).size()
                 .rename("assessments").reset_index()
                 .rename(columns={MODEL: "model_flagged"}).assign(rule_flagged=lambda d: ~d["model_flagged"]))

def pick(rows, group, n=None, by=None, ascending=False):
    """Select review cases from one group, optionally the top n by a column (ties broken by assessment ID)."""
    rows = rows.sort_values([by, "assessment_id"], ascending=[ascending, True]) if by else rows
    return (rows.head(n) if n else rows).assign(review_group=group)

band = early[early["in_boundary_band"]].assign(distance=lambda d: (d["risk_probability"] - MODEL_THRESHOLD).abs())
cases = pd.concat([
    pick(early[early["model_outcome"] == "FN"], "1 missed at-risk (all)", by="risk_probability"),
    pick(early[early["model_outcome"] == "FP"], "2 highest-scoring false positives", n=5, by="risk_probability"),
    pick(band[band[MODEL]], "3 boundary band, just above threshold", n=3, by="distance", ascending=True),
    pick(band[~band[MODEL]], "4 boundary band, just below threshold", n=3, by="distance", ascending=True),
    pick(early[early["model_rule_disagree"] & ~early[MODEL] & ~early["at_risk"]],
         "5 rule flagged, model did not, not at risk", n=3, by="risk_probability", ascending=True),
    pick(early[early["model_outcome"] == "TP"], "6 correct: highest-scoring true positives", n=2, by="risk_probability"),
])
COLUMNS = ["review_group", "assessment_id", "request_id", "checkpoint", "at_risk", "risk_probability", MODEL, RULE,
           "business_days_until_start", "current_status", "business_days_in_current_status", "team_open_workload",
           "task_type", "hiring_department", "employee_type", "intake_channel", "info_issue_at_intake",
           "sim_base_processing_days", "sim_hidden_delay_days", "sim_delay_reason", "sim_workload_delay_days"]
error_review = cases[COLUMNS].round(3).reset_index(drop=True)
error_review.to_csv(OUTPUT_DIR / "error_review_cases.csv", index=False)

display(group_sizes.to_frame())
display(disagreements)
error_review

,assessments
Missed at-risk (model false negatives),6
Model false positives,95
Boundary-review band (0.063 to 0.251),128
Model and global rule disagree,78


,at_risk,model_flagged,assessments,rule_flagged
0,False,False,63,True
1,False,True,9,False
2,True,False,4,True
3,True,True,2,False


,review_group,assessment_id,request_id,checkpoint,at_risk,risk_probability,logistic_regression,days_until_start_rule,business_days_until_start,current_status,...,team_open_workload,task_type,hiring_department,employee_type,intake_channel,info_issue_at_intake,sim_base_processing_days,sim_hidden_delay_days,sim_delay_reason,sim_workload_delay_days
0,1 missed at-risk (all),ASM-02388,REQ-01922,checkpoint_1,True,0.123,False,True,6,open,...,3,orientation_scheduling,Transit,temporary,email,False,2,3,staff_absence,0
1,1 missed at-risk (all),ASM-00191,REQ-00177,checkpoint_1,True,0.101,False,True,6,open,...,16,system_access,Parks,temporary,web_form,False,4,4,vendor_delay,0
2,1 missed at-risk (all),ASM-02557,REQ-02052,checkpoint_1,True,0.097,False,False,8,open,...,3,payroll_setup,Finance,permanent,web_form,False,3,6,staff_absence,0
3,1 missed at-risk (all),ASM-02569,REQ-02052,checkpoint_2,True,0.090,False,True,5,in_progress,...,3,payroll_setup,Finance,permanent,web_form,False,3,6,staff_absence,0
4,1 missed at-risk (all),ASM-00355,REQ-00263,checkpoint_2,True,0.044,False,True,5,in_progress,...,5,workplace_access,Recreation,permanent,web_form,False,5,5,backlog,0
5,1 missed at-risk (all),ASM-00285,REQ-00263,checkpoint_1,True,0.011,False,False,11,open,...,6,workplace_access,Recreation,permanent,web_form,False,5,5,backlog,0
6,2 highest-scoring false positives,ASM-02332,REQ-01875,checkpoint_1,False,0.732,True,True,4,open,...,13,payroll_setup,Recreation,temporary,web_form,False,2,0,none,0
7,2 highest-scoring false positives,ASM-01286,REQ-01045,checkpoint_2,False,0.691,True,True,5,open,...,43,equipment_setup,Parks,seasonal,phone,False,3,0,none,1
8,2 highest-scoring false positives,ASM-02186,REQ-01756,checkpoint_1,False,0.674,True,True,4,open,...,5,payroll_setup,Recreation,temporary,web_form,False,2,0,none,0
9,2 highest-scoring false positives,ASM-01280,REQ-01045,checkpoint_1,False,0.635,True,True,6,open,...,48,equipment_setup,Parks,seasonal,phone,False,3,0,none,1


## C. Simulation-only delays by outcome

This section checks whether errors are associated with the simulation's delays. The first table shows, for each model and global-rule outcome, how often the task had a hidden delay or a workload delay. The second compares recall among at-risk early-warning assessments with and without a hidden delay.

The two delays differ. **Hidden delays** are extra business days that, by design, are not known at assessment time. **Workload delays** come from the responsible team having more open tasks than its capacity when the task was assigned; the model receives `team_open_workload` as an input, which may carry information related to this delay. Only hidden delays are unavailable to the model by design.

Percentages are calculated across assessments, while unique task counts show how much repetition is present. These are simulation-only audit fields used after prediction. An association between hidden delays and missed cases is consistent with some delays being unobservable at assessment time, but it does not establish why any individual prediction failed. Not every task with a hidden delay finished late. The hidden-delay groups are small: among at-risk early-warning assessments, 32 assessments from 19 tasks had hidden delays, so a few cases can change the percentages substantially.

In [4]:
def delay_profile(rows):
    """How often an outcome group had simulation-only delays, with group sizes."""
    hidden = rows["sim_hidden_delay_days"] > 0
    return pd.Series({
        "assessments": len(rows),
        "tasks": rows["request_id"].nunique(),
        "with_hidden_delay": int(hidden.sum()),
        "with_hidden_delay_%": hidden.mean() * 100,
        "median_hidden_delay_days_when_present": rows.loc[hidden, "sim_hidden_delay_days"].median(),
        "with_workload_delay_%": (rows["sim_workload_delay_days"] > 0).mean() * 100,
    })

delay_by_outcome = pd.concat({
    method: early.groupby(f"{prefix}_outcome").apply(delay_profile, include_groups=False).reindex(["TP", "FN", "FP", "TN"])
    for method, prefix in [("model", "model"), ("global_rule", "rule")]
}, names=["method", "outcome"])
delay_by_outcome.round(1).to_csv(OUTPUT_DIR / "delays_by_outcome.csv")

# Among at-risk assessments: recall with and without a hidden delay
at_risk = early[early["at_risk"]].assign(hidden_delay=lambda d: d["sim_hidden_delay_days"] > 0)
recall_by_hidden_delay = pd.concat({
    method: at_risk.groupby("hidden_delay").agg(at_risk_assessments=(flag, "size"), tasks=("request_id", "nunique"),
                                                 flagged=(flag, "sum"), recall=(flag, "mean"))
    for method, flag in [("model", MODEL), ("global_rule", RULE)]
}, names=["method", "hidden_delay"])
recall_by_hidden_delay.round(3).to_csv(OUTPUT_DIR / "recall_by_hidden_delay.csv")

display(delay_by_outcome.round(1))
recall_by_hidden_delay.round(3)

assessments  tasks  with_hidden_delay  \
method      outcome                                          
model       TP              78.0   62.0               26.0   
            FN               6.0    4.0                6.0   
            FP              95.0   67.0                7.0   
            TN             326.0  286.0               55.0   
global_rule TP              80.0   64.0               29.0   
            FN               4.0    4.0                3.0   
            FP             149.0  107.0               14.0   
            TN             272.0  272.0               48.0   

                     with_hidden_delay_%  \
method      outcome                        
model       TP                      33.3   
            FN                     100.0   
            FP                       7.4   
            TN                      16.9   
global_rule TP                      36.2   
            FN                      75.0   
            FP                       9.4   
            TN                      17.6   

                     median_hidden_delay_days_when_present  \
method      outcome                                          
model       TP                                         4.0   
            FN                                         5.0   
            FP                                         1.0   
            TN                                         3.0   
global_rule TP                                         4.0   
            FN                                         5.0   
            FP                                         3.0   
            TN                                         3.0   

                     with_workload_delay_%  
method      outcome                         
model       TP                        38.5  
            FN                         0.0  
            FP                        26.3  
            TN                        16.9  
global_rule TP                        36.2  
            FN                        25.0  
            FP                        16.8  
            TN                        20.2

at_risk_assessments  tasks  flagged  recall
method      hidden_delay                                             
model       False                          52     45       52   1.000
            True                           32     19       26   0.812
global_rule False                          52     45       51   0.981
            True                           32     19       29   0.906

## D. Slice results

Results are compared across four non-sensitive operational slices: task type, intake channel, checkpoint, and employee type (a job category, not a personal attribute). For each group, the table shows assessment, task, and employee counts, the number at risk with the tasks and employees behind them, the number flagged, recall, precision, and the share flagged, for the model and the global days rule.

All slice results are reported. Recall estimates based on fewer than 20 at-risk assessments and precision estimates based on fewer than 20 flagged assessments are marked as having limited support; others are marked 20+, meaning they meet the reporting minimum. This exploratory rule highlights small denominators; it does not establish statistical reliability. Repeated assessments of one task and several tasks for one employee are related. Task and employee counts help show the extent of clustering; they do not establish the number of independent observations. One employee can appear in more than one group within a slice, for example with tasks of different types.

The 90% recall target applies to all early-warning assessments, not to each slice separately. A second table shows where the model's flag reduction occurs: global-rule flags minus model flags in each group.

Differences between groups are a reason to investigate, not proof that a method performs differently for every case in that group.

In [5]:
SLICES = ["task_type", "intake_channel", "checkpoint", "employee_type"]
MIN_SUPPORT = 20   # practical reporting rule for small denominators, not a reliability guarantee

def slice_measures(rows, flag):
    """Counts and measures for one method within one slice group."""
    y, p = rows["at_risk"], rows[flag]
    tp = int((p & y).sum())
    return pd.Series({
        "assessments": len(rows), "tasks": rows["request_id"].nunique(), "employees": rows["employee_id"].nunique(),
        "at_risk": int(y.sum()), "at_risk_tasks": rows.loc[y, "request_id"].nunique(),
        "employees_with_at_risk": rows.loc[y, "employee_id"].nunique(),
        "flagged": int(p.sum()), "TP": tp,
        "recall": tp / y.sum() if y.sum() else np.nan,
        "precision": tp / p.sum() if p.sum() else np.nan,
        "flagged_%": p.mean() * 100,
        "recall_support": "limited" if y.sum() < MIN_SUPPORT else "20+",
        "precision_support": "limited" if p.sum() < MIN_SUPPORT else "20+",
    })

slice_rows = []
for slice_name in SLICES:
    for group, rows in early.groupby(slice_name):
        for method, flag in [("model", MODEL), ("global_rule", RULE)]:
            slice_rows.append(slice_measures(rows, flag).rename((slice_name, group, method)))
slice_results = pd.DataFrame(slice_rows)
slice_results.index.names = ["slice", "group", "method"]
slice_results.round(3).to_csv(OUTPUT_DIR / "slice_results.csv")

# Where the model's flag reduction occurs: global-rule flags minus model flags in each group
flag_difference = (slice_results["flagged"].unstack("method")[["global_rule", "model"]]
                   .assign(rule_minus_model=lambda d: d["global_rule"] - d["model"]))
flag_difference.to_csv(OUTPUT_DIR / "flag_difference_by_group.csv")

pd.set_option("display.max_rows", 100)
display(slice_results.round(3))
flag_difference

assessments  tasks  \
slice          group                  method                            
task_type      equipment_setup        model                 73     53   
                                      global_rule           73     53   
               orientation_scheduling model                 97     81   
                                      global_rule           97     81   
               payroll_setup          model                128     99   
                                      global_rule          128     99   
               system_access          model                121     96   
                                      global_rule          121     96   
               workplace_access       model                 86     68   
                                      global_rule           86     68   
intake_channel email                  model                145    116   
                                      global_rule          145    116   
               phone                  model                 97     75   
                                      global_rule           97     75   
               web_form               model                263    206   
                                      global_rule          263    206   
checkpoint     checkpoint_1           model                397    397   
                                      global_rule          397    397   
               checkpoint_2           model                108    108   
                                      global_rule          108    108   
employee_type  permanent              model                220    201   
                                      global_rule          220    201   
               seasonal               model                157    105   
                                      global_rule          157    105   
               temporary              model                128     91   
                                      global_rule          128     91   

                                                   employees  at_risk  \
slice          group                  method                            
task_type      equipment_setup        model               53       17   
                                      global_rule         53       17   
               orientation_scheduling model               81       10   
                                      global_rule         81       10   
               payroll_setup          model               99       33   
                                      global_rule         99       33   
               system_access          model               96       12   
                                      global_rule         96       12   
               workplace_access       model               68       12   
                                      global_rule         68       12   
intake_channel email                  model               79       18   
                                      global_rule         79       18   
               phone                  model               52       20   
                                      global_rule         52       20   
               web_form               model               94       46   
                                      global_rule         94       46   
checkpoint     checkpoint_1           model              100       64   
                                      global_rule        100       64   
               checkpoint_2           model               41       20   
                                      global_rule         41       20   
employee_type  permanent              model               51        6   
                                      global_rule         51        6   
               seasonal               model               27       52   
                                      global_rule         27       52   
               temporary              model               22       26   
                                   

method                                 global_rule  model  rule_minus_model
slice          group                                                       
checkpoint     checkpoint_1                    121    119                 2
               checkpoint_2                    108     54                54
employee_type  permanent                        24     13                11
               seasonal                        127    104                23
               temporary                        78     56                22
intake_channel email                            66     49                17
               phone                            49     41                 8
               web_form                        114     83                31
task_type      equipment_setup                  39     34                 5
               orientation_scheduling           43     21                22
               payroll_setup                    55     56                -1
               system_access                    52     35                17
               workplace_access                 40     27                13

## E. Interpretation

This interpretation follows the error-analysis questions from the course, adapted for yes/no predictions: which predictions were most confidently incorrect, which direction the errors go, whether patterns differ between groups, whether the groups are large enough, what might explain the pattern, and what further evidence would be needed.

**Which predictions were most confidently incorrect?** The model missed six at-risk validation assessments from four tasks, one of each task type except equipment setup. Their estimated risks ranged from 0.011 to 0.123. They are not equally clear misses: REQ-01922, at 0.123, was just below the 0.125 threshold, while REQ-00263, at 0.011, was far below it. The highest-scoring false positives had estimated risks of 0.619 to 0.732; these flagged cases had 4 to 6 weekday counts until start, and none had a hidden delay.

**Which direction do the errors go?** The model made 95 false positives and 6 false negatives. The chosen threshold prioritizes recall, accepting unnecessary flags to reduce missed at-risk assessments. The larger number of false positives also reflects that most assessments are not at risk. Compared with the global days rule, the model avoided 63 false positives the rule made and made 9 the rule did not, while the rule caught 4 at-risk assessments the model missed and the model caught 2 the rule missed. The clearest avoided false positives were checkpoint-2 assessments already in progress, with estimated risks of 0.015 to 0.028.

**Are errors associated with the simulation's delays?** All six model false negatives had a hidden delay of 3 to 6 business days, compared with 33.3% of the model's true positives. Among at-risk early-warning assessments, the model flagged all 52 without a hidden delay and 26 of the 32 with one (81.2%); the global rule flagged 98.1% and 90.6%. The model's misses were therefore concentrated in hidden-delay cases. Hidden delays are not known at assessment time by design, which is consistent with some misses being hard to anticipate from the available inputs, but this does not establish why any individual prediction failed. Workload delays differ: the model receives team open workload as an input.

**Do patterns differ between groups, and are the groups large enough?** Among slices that meet the 20 at-risk reporting minimum, the model's recall ranged from 0.891 (web form) to 1.000 (phone and seasonal employees), and it flagged a smaller share than the global rule in every group except payroll setup. At checkpoint 2, the global rule flagged all 108 assessments with 18.5% precision, while the model flagged 54 with 90.0% recall and 33.3% precision; 54 of the model's 56 fewer flags came from checkpoint 2. Web-form recall of 89.1% is below 90%, but the 90% target applies to all early-warning assessments, not to each slice. Permanent employees had the lowest model recall (2 of 6), but those six at-risk assessments come from only three tasks and three employees, so this is a small-sample concern, not evidence of consistently poor performance for that group.

**What further evidence would be needed?** Larger validation groups, especially for permanent employees and individual task types; a time-based check, because the splits cover overlapping dates; and, in a real setting, information about delays that is currently unavailable at assessment time. Exercise 09 will compare flagging thresholds and define a separate review policy using validation data before final testing. The current boundary band is exploratory.

All results describe methods applied to simulated outcomes, not real employees or HR performance.